In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Silver_Order_Items
# Bronze -> Silver
# ============================================================

SOURCE_TABLE = "bronze_order_items"
ORDERS_TABLE = "silver_orders"
PRODUCTS_TABLE = "silver_products"
SILVER_TABLE = "silver_order_items"
QUARANTINE_TABLE = "quarantine_order_items"

# ------------------------------------------------------------
# 1. Read Bronze
# ------------------------------------------------------------

items = spark.table(SOURCE_TABLE)

orders = (
    spark.table(ORDERS_TABLE)
    .select("order_id")
    .dropDuplicates()
)

products = (
    spark.table(PRODUCTS_TABLE)
    .select("product_id")
    .dropDuplicates()
)

# ------------------------------------------------------------
# 2. Standardize types
# ------------------------------------------------------------

items = (
    items
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)"))
)

# ------------------------------------------------------------
# 3. Referential integrity
# ------------------------------------------------------------

items = (
    items
    .join(
        orders.withColumn("order_exists", F.lit(True)),
        on="order_id",
        how="left"
    )
    .join(
        products.withColumn("product_exists", F.lit(True)),
        on="product_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------

df_checked = (
    items
    .withColumn(
        "dq_reason",
        F.when(
            F.col("order_item_id").isNull(),
            F.lit("NULL_ORDER_ITEM_ID")
        )
        .when(
            F.col("order_id").isNull(),
            F.lit("NULL_ORDER_ID")
        )
        .when(
            F.col("order_exists").isNull(),
            F.lit("INVALID_ORDER_REFERENCE")
        )
        .when(
            F.col("product_id").isNull(),
            F.lit("NULL_PRODUCT_ID")
        )
        .when(
            F.col("product_exists").isNull(),
            F.lit("INVALID_PRODUCT_REFERENCE")
        )
        .when(
            F.col("quantity").isNull(),
            F.lit("NULL_QUANTITY")
        )
        .when(
            F.col("quantity") <= 0,
            F.lit("INVALID_QUANTITY")
        )
        .when(
            F.col("unit_price").isNull(),
            F.lit("NULL_UNIT_PRICE")
        )
        .when(
            F.col("unit_price") < 0,
            F.lit("NEGATIVE_UNIT_PRICE")
        )
        .otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 5. Quarantine
# ------------------------------------------------------------

df_quarantine = (
    df_checked
    .filter(F.col("dq_reason").isNotNull())
    .drop("order_exists", "product_exists")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_quarantine.write.mode("overwrite").format("delta").saveAsTable(
    QUARANTINE_TABLE
)

# ------------------------------------------------------------
# 6. Valid records
# ------------------------------------------------------------

df_valid = (
    df_checked
    .filter(F.col("dq_reason").isNull())
    .drop("dq_reason", "order_exists", "product_exists")
)

# ------------------------------------------------------------
# 7. Deduplicate
# ------------------------------------------------------------

window = Window.partitionBy("order_item_id").orderBy(
    F.col("order_id"),
    F.col("product_id")
)

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# ------------------------------------------------------------
# 8. Write Silver
# ------------------------------------------------------------

df_dedup.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_TABLE
)

print(f"Silver table created: {SILVER_TABLE}")
print(f"Valid records: {df_dedup.count()}")
print(f"Quarantined records: {df_quarantine.count()}")

StatementMeta(, 876fa136-19e9-476d-a069-2f2840fc4605, 3, Finished, Available, Finished, False)

Silver table created: silver_order_items
Valid records: 119569
Quarantined records: 431
